# 12 — Rigorous encoder audit

Compare **encoder-family** toxic classifiers on the same Jigsaw slices:

| key | role |
|---|---|
| `toxic-bert` | in-domain BERT teacher (`unitary/toxic-bert`) |
| `distilbert-tox` | off-the-shelf DistilBERT toxicity model (`martin-ha/toxic-comment-model`, other data) |
| `distilbert-student` | our soft-KD student (same-data distillation from toxic-bert) |

Same `predict_toxic_proba` harness as notebooks 03/11. Missing local student weights are skipped with a message.
Set `QUICK_MODE = False` for submission-quality sizes.


In [ ]:
%matplotlib inline
import json, sys, time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

THRESHOLD = 0.5

_src = None
for _p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (_p / "src" / "paths.py").is_file():
        _src = str(_p / "src")
        break
if _src is None:
    raise FileNotFoundError("Cannot find DeepProjectHate/src/paths.py")
if _src not in sys.path:
    sys.path.insert(0, _src)

for _m in (
    "paths", "model_registry", "roster", "splits", "fairness",
    "perturb", "intervention", "device", "distill",
):
    sys.modules.pop(_m, None)

from device import get_device
from fairness import CONTROL_TEXTS, build_identity_probe_df, control_fpr, per_group_fpr
from intervention import mask_terms
from model_registry import ALL_SPECS, load_scorer
from paths import PROCESSED_DIR, PROJECT_ROOT, SEED
from perturb import PERTURBATIONS
from roster import model_ready
from splits import get_split_frame, sample_split_frame

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
device = get_device()
print("ROOT", PROJECT_ROOT)
print("device", device)

QUICK_MODE = True
EVAL_N = 100 if QUICK_MODE else 500
ROBUST_N = 60 if QUICK_MODE else 200
KEYS = ["toxic-bert", "distilbert-tox", "distilbert-student"]
print(f"QUICK_MODE={QUICK_MODE} EVAL_N={EVAL_N}")
print("Roster:", KEYS)

def binary_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    out = {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
    }
    if len(np.unique(y_true)) > 1:
        out["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        out["pr_auc"] = float(average_precision_score(y_true, y_prob))
    else:
        out["roc_auc"] = float("nan")
        out["pr_auc"] = float("nan")
    return out


def score_split(df, scorer):
    texts = df["comment_text"].astype(str).tolist()
    y = df["toxic"].astype(int).values
    t0 = time.perf_counter()
    p = np.asarray(scorer.predict_toxic_proba(texts), dtype=float)
    ms = 1000.0 * (time.perf_counter() - t0) / max(len(texts), 1)
    m = binary_metrics(y, p)
    m["ms_per_comment"] = round(ms, 3)
    return p, y, m


def load_ready(keys):
    scorers = {}
    for key in keys:
        ok, detail = model_ready(key) if key in (
            "toxic-bert", "tfidf-svm", "distilgpt2-d1", "distilgpt2-d1-ft", "distilbert-student"
        ) else (True, "registry")
        # Extra checks for keys not in FAIR_ROSTER helpers
        if key == "distilbert-tox":
            ok, detail = True, "HF hub martin-ha/toxic-comment-model"
        if key == "distilgpt2-d2":
            ok, detail = model_ready("distilgpt2-d1")[0], "same DistilGPT-2 weights as d1"
        if not ok:
            print(f"  [SKIP] {key}: {detail}")
            continue
        try:
            scorers[key] = load_scorer(ALL_SPECS[key], device=device)
            n = getattr(scorers[key], "n_params", None)
            print(f"  [OK] {key}: {detail} | params={n}")
        except Exception as exc:  # noqa: BLE001
            print(f"  [SKIP] {key}: load failed: {exc}")
    return scorers


## 1. Load models (skip if missing)


In [ ]:
scorers = load_ready(KEYS)
assert scorers, "No encoder models loaded"
val = sample_split_frame("val", n=EVAL_N, seed=SEED)
test = sample_split_frame("test", n=EVAL_N, seed=SEED)
print(f"val_n={len(val)} test_n={len(test)}")


## 2. Held-out metrics (val + test)


In [ ]:
metric_rows = []
probs = {}
for key, scorer in scorers.items():
    for split_name, df in (("val", val), ("test", test)):
        p, y, m = score_split(df, scorer)
        probs[(key, split_name)] = (p, y)
        metric_rows.append({"model": key, "split": split_name, **m,
                            "n_params": getattr(scorer, "n_params", None)})
        print(f"{key:20s} {split_name}: F1={m['f1']:.3f} PR-AUC={m['pr_auc']:.3f} ms={m['ms_per_comment']:.1f}")
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))

test_m = metrics_df[metrics_df["split"] == "test"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=test_m, x="model", y="f1", ax=axes[0], color="steelblue")
axes[0].set_title("Test F1"); axes[0].tick_params(axis="x", rotation=20); axes[0].set_ylim(0, 1)
sns.barplot(data=test_m, x="model", y="ms_per_comment", ax=axes[1], color="darkorange")
axes[1].set_title("Latency (ms/comment)"); axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()


## 3. Confusion + ROC (test)


In [ ]:
n = len(scorers)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.5))
if n == 1:
    axes = [axes]
for ax, key in zip(axes, scorers):
    p, y = probs[(key, "test")]
    cm = confusion_matrix(y, (p >= THRESHOLD).astype(int))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["pred0", "pred1"], yticklabels=["gold0", "gold1"])
    ax.set_title(key)
plt.tight_layout(); plt.show()

plt.figure(figsize=(6, 5))
for key in scorers:
    p, y = probs[(key, "test")]
    if len(np.unique(y)) < 2:
        continue
    fpr, tpr, _ = roc_curve(y, p)
    auc = roc_auc_score(y, p)
    plt.plot(fpr, tpr, label=f"{key} ({auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Test ROC"); plt.legend()
plt.tight_layout(); plt.show()


## 4. Error analysis (per model, top FP / FN)


In [ ]:
for key in scorers:
    p, y = probs[(key, "test")]
    err = test.copy()
    err["prob"] = p
    err["pred"] = (p >= THRESHOLD).astype(int)
    err["gold"] = y
    fp = err[(err.gold == 0) & (err.pred == 1)].sort_values("prob", ascending=False).head(5)
    fn = err[(err.gold == 1) & (err.pred == 0)].sort_values("prob").head(5)
    print(f"\n=== {key} === FP={((err.gold==0)&(err.pred==1)).sum()} FN={((err.gold==1)&(err.pred==0)).sum()}")
    display(fp[["comment_text", "prob"]].assign(comment_text=lambda d: d.comment_text.str.slice(0, 160)))
    display(fn[["comment_text", "prob"]].assign(comment_text=lambda d: d.comment_text.str.slice(0, 160)))


## 5. Fairness (identity FPR)


In [ ]:
probe = build_identity_probe_df()
fair_rows = []
group_tables = {}
for key, scorer in scorers.items():
    pp = scorer.predict_toxic_proba(probe["text"].tolist())
    g = per_group_fpr(probe, pp, threshold=THRESHOLD)
    ctrl = control_fpr(scorer.predict_toxic_proba(CONTROL_TEXTS), threshold=THRESHOLD)
    group_tables[key] = g
    fair_rows.append({
        "model": key,
        "overall_fpr": float(g["fpr"].mean()),
        "control_fpr": ctrl["fpr"],
        "max_group_fpr": float(g["fpr"].max()),
    })
    print(f"{key}: overall FPR={fair_rows[-1]['overall_fpr']:.3f} control={ctrl['fpr']:.3f}")
fair_df = pd.DataFrame(fair_rows)
display(fair_df.round(3))

ax = sns.barplot(data=fair_df, x="model", y="overall_fpr", color="crimson")
ax.set_title("Overall identity FPR"); ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()


## 6. Robustness (evasion)


In [ ]:
toxic_pool = get_split_frame("test")
toxic_pool = toxic_pool[toxic_pool["toxic"] == 1]
if len(toxic_pool) > ROBUST_N:
    toxic_pool = toxic_pool.sample(ROBUST_N, random_state=SEED)
base_texts = toxic_pool["comment_text"].astype(str).tolist()

rob_rows = []
for key, scorer in scorers.items():
    base_p = np.asarray(scorer.predict_toxic_proba(base_texts))
    base_flag = base_p >= THRESHOLD
    for pname, fn in PERTURBATIONS.items():
        pert = [fn(t) for t in base_texts]
        pp = np.asarray(scorer.predict_toxic_proba(pert))
        still = pp >= THRESHOLD
        idx = np.where(base_flag)[0]
        evasion = float((~still[idx]).mean()) if len(idx) else float("nan")
        rob_rows.append({"model": key, "perturbation": pname, "evasion_rate": evasion,
                         "detection_rate": float(still.mean())})
rob_df = pd.DataFrame(rob_rows)
display(rob_df.pivot_table(index="perturbation", columns="model", values="evasion_rate").round(3))

g = sns.catplot(data=rob_df, x="model", y="evasion_rate", hue="perturbation",
                kind="bar", height=4, aspect=1.6)
g.set_xticklabels(rotation=20)
g.fig.suptitle("Evasion rate", y=1.02)
plt.show()


## 7. Inference masking


In [ ]:
mask_rows = []
for key, scorer in scorers.items():
    p_id = np.asarray(scorer.predict_toxic_proba(probe["text"].tolist()))
    p_mask = np.asarray(scorer.predict_toxic_proba(probe["text"].map(mask_terms).tolist()))
    mask_rows.append({
        "model": key,
        "fpr_with_identity": float((p_id >= THRESHOLD).mean()),
        "fpr_masked": float((p_mask >= THRESHOLD).mean()),
        "mean_prob_with_identity": float(p_id.mean()),
        "mean_prob_masked": float(p_mask.mean()),
    })
mask_df = pd.DataFrame(mask_rows)
display(mask_df.round(3))
plot_df = mask_df.melt(id_vars=["model"], value_vars=["fpr_with_identity", "fpr_masked"],
                       var_name="stage", value_name="fpr")
sns.barplot(data=plot_df, x="model", y="fpr", hue="stage")
plt.title("Inference masking FPR"); plt.xticks(rotation=20)
plt.tight_layout(); plt.show()


## 8. Cache-first side-by-side (optional)


In [ ]:
comp_path = PROCESSED_DIR / "model_comparison.json"
if comp_path.exists():
    payload = json.loads(comp_path.read_text(encoding="utf-8"))
    rows = []
    for key in KEYS:
        b = payload.get("models", {}).get(key)
        if not b:
            print(f"[CACHE MISS] {key}")
            continue
        t = b.get("splits", {}).get("test", {})
        rows.append({"model": key, "test_f1": t.get("f1"), "ms": t.get("ms_per_comment"),
                     "n_params": b.get("n_params"), "source": "cache"})
    display(pd.DataFrame(rows).round(3))
else:
    print("No", comp_path)


## 9. Save JSON


In [ ]:
out = {
    "seed": SEED,
    "quick_mode": QUICK_MODE,
    "eval_n": EVAL_N,
    "keys": KEYS,
    "loaded": list(scorers),
    "metrics": metric_rows,
    "fairness": fair_rows,
    "robustness": rob_df.to_dict(orient="records"),
    "masking": mask_rows,
}
path = PROCESSED_DIR / "encoder_rigorous_audit.json"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2), encoding="utf-8")
print("Wrote", path)
